In [1]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl

In [2]:
quartus_bin = r"C:\altera_lite\25.1std\quartus\bin64"

# Compile
subprocess.run(
    [f"{quartus_bin}\\quartus_sh.exe", "--flow", "compile", "original_generic_tree_16_n"],
    cwd=r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\arvorev2",
    check=True
)

# Timing
subprocess.run(
    [f"{quartus_bin}\\quartus_sta.exe", "-t", "script.tcl"],
    cwd=r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\arvorev2",
    check=True
)

CompletedProcess(args=['C:\\altera_lite\\25.1std\\quartus\\bin64\\quartus_sta.exe', '-t', 'script.tcl'], returncode=0)

## Diretorios

In [2]:
quartus_bin = r"C:\altera_lite\25.1std\quartus\bin64"
project_dir = r"C:\Users\Rafa\Desktop\HCR\FABIAN\somador2026\arvorev2"
vhdl_file   = project_dir + r"\vhdl\sum16xn.vhd"
report_file = project_dir + r"\io_paths.txt"



## Functions

In [3]:
# =========================
# altera n no VHDL
# =========================
def set_n(n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(r"n\s*:\s*integer\s*:=\s*\d+", f"n : integer := {n}", text)

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)

# =========================
# roda quartus
# =========================
def run_quartus():
    # limpa cache (importante!)
    shutil.rmtree(project_dir + "\\db", ignore_errors=True)
    shutil.rmtree(project_dir + "\\incremental_db", ignore_errors=True)

    subprocess.run(
        [quartus_bin + "\\quartus_sh.exe", "--flow", "compile", "original_generic_tree_16_n"],
        cwd=project_dir,
        check=True
    )

    subprocess.run(
        [quartus_bin + "\\quartus_sta.exe", "-t", "script.tcl"],
        cwd=project_dir,
        check=True
    )

# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)
                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths).dropna()
    return df

# =========================
#  calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }

def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            # Delay total
            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            # From / To
            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            # IC total
            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            # CELL total
            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [4]:
n_values = [2048, 4096]

resultados = []

for n in tqdm(n_values):
    try:
        print(f"\n Rodando n = {n}")

        set_n(n)
        run_quartus()

        df = parse_paths(report_file)

        if df.empty:
            raise ValueError("Nenhum path encontrado")

        metrics = compute_metrics(df)

        cp = extract_critical_path(project_dir + r"\worst_path.txt")

        resultados.append({
            "n": n,

            "Delay_CP": cp["delay"],
            "IC_CP": cp["ic"],
            "CELL_CP": cp["cell"],

            "Delay_MAX": metrics["Delay_MAX"],
            "IC_MAX": metrics["IC_MAX"],
            "CELL_MAX": metrics["CELL_MAX"],

            "Delay_MEAN": metrics["Delay_MEAN"],
            "IC_MEAN": metrics["IC_MEAN"],
            "CELL_MEAN": metrics["CELL_MEAN"],
        })

    except Exception as e:
        print(f"Erro em n={n}: {e}")

final_df = pd.DataFrame(resultados).sort_values("n")

print("\nResultado final:")
print(final_df)

final_df.to_csv("timing_vs_n.csv", index=False)
final_df.to_excel("timing_vs_n.xlsx", index=False)

  0%|          | 0/2 [00:00<?, ?it/s]


 Rodando n = 2048


 50%|█████     | 1/2 [03:27<03:27, 207.57s/it]

Erro em n=2048: Command '['C:\\altera_lite\\25.1std\\quartus\\bin64\\quartus_sh.exe', '--flow', 'compile', 'original_generic_tree_16_n']' returned non-zero exit status 3.

 Rodando n = 4096


100%|██████████| 2/2 [17:50<00:00, 535.25s/it]

Erro em n=4096: Command '['C:\\altera_lite\\25.1std\\quartus\\bin64\\quartus_sh.exe', '--flow', 'compile', 'original_generic_tree_16_n']' returned non-zero exit status 3.


KeyError: 'n'